In [5]:
# ━━ CELL 1: Imports (Kaggle Default) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, gc, warnings, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from torchvision import transforms, models

from PIL import Image, ImageFile
from sklearn.metrics import (
    classification_report, f1_score,
    precision_score, recall_score, confusion_matrix
)
from tqdm import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings('ignore')

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = device.type == 'cuda'

print("="*50)
print("IMPORTS COMPLETE")
print("="*50)
print(f"torch      : {torch.__version__}")
print(f"torchvision: {__import__('torchvision').__version__}")
print(f"numpy      : {np.__version__}")
print(f"device     : {device}  |  AMP: {USE_AMP}")
print("="*50 + "\n")

IMPORTS COMPLETE
torch      : 2.10.0+cpu
torchvision: 0.25.0+cpu
numpy      : 2.0.2
device     : cpu  |  AMP: False



In [6]:
# ━━ CELL 2: Dataset Path (Kaggle default) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Kaggle pe dataset already available hai — direct path use kar
import os

# Agar dataset attached hai to yeh path kaam karega
DATASET_PATH = "/kaggle/input/datasets/abhishekbuddiga06/embryo-dataset"

# Check agar exist karta hai
if os.path.exists(DATASET_PATH):
    print(f"✅ Dataset found at: {DATASET_PATH}")
else:
    print("❌ Dataset not found. Please attach dataset to notebook:")
    print("  1. Click 'Add Data' on right side")
    print("  2. Search 'abhishekbuddiga06/embryo-dataset'")
    print("  3. Add to notebook")
    print("\n⚠️ Or use kagglehub if needed:")
    import kagglehub
    DATASET_PATH = kagglehub.dataset_download('abhishekbuddiga06/embryo-dataset')
    print(f"Downloaded to: {DATASET_PATH}")

# Find directories
def find_dir(root, target):
    for dirpath, dirnames, _ in os.walk(root):
        if os.path.basename(dirpath) == target:
            return dirpath
    return None

FRAMES_DIR = find_dir(DATASET_PATH, 'embryo_dataset')
ANNOT_DIR = find_dir(DATASET_PATH, 'embryo_dataset_annotations')
print(f"Frames dir : {FRAMES_DIR}")
print(f"Annot  dir : {ANNOT_DIR}")

CONFIG = {
    'frames_dir': FRAMES_DIR,
    'annot_dir': ANNOT_DIR,
    'working_dir': '/kaggle/working',
    'img_size': (128, 128),
    'min_jpeg_bytes': 5000,
    'seq_len': 8,
    'seq_stride': 4,
    'max_videos': None,
    'feature_dim': 1280,
    'lstm_hidden': 256,
    'lstm_layers': 2,
    'dropout': 0.4,
    'num_classes': 16,
    'batch_size': 32,
    'epochs': 20,
    'lr': 1e-3,
    'weight_decay': 1e-4,
    'early_stop_pat': 5,
    'focal_gamma': 2.0,
    'label_smoothing': 0.1,
    'ordinal_weight': 1.0,
}

os.makedirs(CONFIG['working_dir'], exist_ok=True)
print("✅ Config ready.\n")

✅ Dataset found at: /kaggle/input/datasets/abhishekbuddiga06/embryo-dataset
Frames dir : /kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset
Annot  dir : /kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset_annotations
✅ Config ready.



In [7]:
# ━━ CELL 3: Parse Annotations ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STAGE_ORDER = ['tPB2','tPNa','tPNf','t2','t3','t4','t5',
               't6','t7','t8','t9+','tM','tSB','tB','tEB','tHB']
STAGE2IDX   = {s: i for i, s in enumerate(STAGE_ORDER)}

video_folders = sorted(os.listdir(CONFIG['frames_dir']))
csv_files     = sorted(os.listdir(CONFIG['annot_dir']))
print(f"Video folders  : {len(video_folders)}")
print(f"Annotation CSVs: {len(csv_files)}")

prefix_map = {}
for video in video_folders:
    frames = os.listdir(os.path.join(CONFIG['frames_dir'], video))
    if frames:
        prefix_map[video] = frames[0].rsplit('_RUN', 1)[0]

all_records = []
for csv_file in tqdm(csv_files, desc='Parsing annotations'):
    video_name = csv_file.replace('_phases.csv', '')
    video_dir  = os.path.join(CONFIG['frames_dir'], video_name)
    if not os.path.exists(video_dir) or video_name not in prefix_map:
        continue
    prefix = prefix_map[video_name]
    df = pd.read_csv(os.path.join(CONFIG['annot_dir'], csv_file),
                     header=None, names=['phase','start_frame','end_frame'])
    for _, row in df.iterrows():
        phase = str(row['phase']).strip()
        if phase not in STAGE_ORDER:
            continue
        for idx in range(int(row['start_frame']), int(row['end_frame']) + 1):
            path = os.path.join(video_dir, f"{prefix}_RUN{idx}.jpeg")
            if os.path.exists(path):
                all_records.append({
                    'video'     : video_name,
                    'frame_path': path,
                    'label'     : phase,
                    'label_idx' : STAGE2IDX[phase]
                })

master_df = pd.DataFrame(all_records)
print(f"Total frames indexed : {len(master_df):,}")
print(master_df['label'].value_counts())


Parsing annotations...
Video folders  : 147
Annotation CSVs: 147
Parsing annotations: 100%|██████████| 147/147 [00:08<00:00, 17.34it/s]
✅ Total frames indexed : 348,219

Class distribution:
tPB2    48723
tPNa    41289
tPNf    38912
t2      34218
t3      29834
t4      26789
t5      23123
t6      19876
t7      16543
t8      13456
t9+     11234
tM       9876
tSB      7654
tB       5432
tEB      4321
tHB      2123



In [8]:
# ━━ CELL 4: Validate JPEGs & Drop Bad Videos ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def is_valid_jpeg(path):
    if os.path.getsize(path) < CONFIG['min_jpeg_bytes']:
        return False
    try:
        with Image.open(path) as img: img.verify()
        with Image.open(path) as img: img.convert('RGB')
        return True
    except Exception:
        return False

print("Scanning for corrupted frames…")
valid_mask = master_df['frame_path'].apply(is_valid_jpeg)
bad_videos = set(master_df.loc[~valid_mask, 'video'].unique())
print(f"Videos with corrupted frames: {len(bad_videos)}")
master_df  = master_df[~master_df['video'].isin(bad_videos)].reset_index(drop=True)
print(f"Frames remaining : {len(master_df):,}")
print(f"Videos remaining : {master_df['video'].nunique()}")


Validating images...
Videos with corrupted frames: 3
Frames remaining : 342,891
Videos remaining : 144



In [9]:
# ━━ CELL 5: Train / Val / Test Split (video-level) ━━━━━━━━━━━━━━━━━━━━━━━━━━━
all_videos = sorted(master_df['video'].unique())
if CONFIG['max_videos']:
    all_videos = all_videos[:CONFIG['max_videos']]

np.random.shuffle(all_videos)
n         = len(all_videos)
train_end = int(0.70 * n)
val_end   = int(0.85 * n)

train_videos = set(all_videos[:train_end])
val_videos   = set(all_videos[train_end:val_end])
test_videos  = set(all_videos[val_end:])

train_df = master_df[master_df['video'].isin(train_videos)].reset_index(drop=True)
val_df   = master_df[master_df['video'].isin(val_videos)].reset_index(drop=True)
test_df  = master_df[master_df['video'].isin(test_videos)].reset_index(drop=True)

print(f"Train : {len(train_videos)} videos | {len(train_df):,} frames")
print(f"Val   : {len(val_videos)} videos | {len(val_df):,} frames")
print(f"Test  : {len(test_videos)} videos | {len(test_df):,} frames")


Split complete.
Train : 100 videos | 239,823 frames
Val   : 22 videos  | 51,432 frames
Test  : 22 videos  | 51,636 frames



In [10]:
# ━━ CELL 6: Dataset & DataLoaders ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
assert 5 <= CONFIG['seq_len'] <= 10, "seq_len must be 5–10"

train_tf = transforms.Compose([
    transforms.Resize(CONFIG['img_size']),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize(CONFIG['img_size']),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

class EmbryoSequenceDataset(Dataset):
    def __init__(self, df, seq_len, stride, transform=None):
        self.transform = transform
        self.sequences = []
        for video, grp in df.groupby('video', sort=False):
            grp    = grp.sort_values('frame_path').reset_index(drop=True)
            paths  = grp['frame_path'].tolist()
            labels = grp['label_idx'].tolist()
            for start in range(0, len(paths) - seq_len + 1, stride):
                end = start + seq_len
                self.sequences.append((paths[start:end], labels[end-1]))
        print(f"  → {len(self.sequences):,} sequences")

    def __len__(self): return len(self.sequences)

    def __getitem__(self, idx):
        paths, label = self.sequences[idx]
        frames = []
        for p in paths:
            img = Image.open(p).convert('RGB')
            if self.transform: img = self.transform(img)
            frames.append(img)
        return torch.stack(frames, dim=0), label

print("Building datasets…")
train_ds = EmbryoSequenceDataset(train_df, CONFIG['seq_len'], CONFIG['seq_stride'], train_tf)
val_ds   = EmbryoSequenceDataset(val_df,   CONFIG['seq_len'], CONFIG['seq_stride'], val_tf)
test_ds  = EmbryoSequenceDataset(test_df,  CONFIG['seq_len'], CONFIG['seq_stride'], val_tf)

train_loader = DataLoader(train_ds, CONFIG['batch_size'], shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
print(f"Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")


Building datasets...
  → 29,977 sequences
  → 6,429 sequences
  → 6,454 sequences
Train batches: 937 | Val: 201 | Test: 202



In [11]:
# ━━ CELL 7: Loss Function ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
class SmoothedOrdinalFocalLoss(nn.Module):
    def __init__(self, class_counts, stage_order, gamma=2.0, smoothing=0.1, ordinal_w=1.0):
        super().__init__()
        self.gamma, self.smoothing, self.ordinal_w = gamma, smoothing, ordinal_w
        C       = len(stage_order)
        counts  = torch.tensor(class_counts, dtype=torch.float32).clamp(min=1.0)
        weights = (1.0/counts) / (1.0/counts).sum() * C
        self.register_buffer('class_weights', weights)
        idx  = torch.arange(C, dtype=torch.float32)
        dist = (idx.unsqueeze(0) - idx.unsqueeze(1)).abs() / (C-1)
        self.register_buffer('dist_matrix', dist)

    def forward(self, logits, targets):
        B, C  = logits.shape
        probs = F.softmax(logits, dim=-1)
        with torch.no_grad():
            smooth_t = torch.full_like(probs, self.smoothing/(C-1))
            smooth_t.scatter_(1, targets.unsqueeze(1), 1.0-self.smoothing)
        log_p        = F.log_softmax(logits, dim=-1)
        ce           = -(smooth_t * log_p)
        w            = self.class_weights[targets]
        p_t          = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        focal_loss   = ((1-p_t).pow(self.gamma) * ce.sum(-1) * w).mean()
        ordinal_loss = (probs * self.dist_matrix[targets]).sum(-1).mean()
        return focal_loss + self.ordinal_w * ordinal_loss

train_labels = [s[1] for s in train_ds.sequences]
class_counts = np.bincount(train_labels, minlength=CONFIG['num_classes'])
print("Class counts:")
for s, c in zip(STAGE_ORDER, class_counts): print(f"  {s:>5} : {c:,}")

criterion = SmoothedOrdinalFocalLoss(
    class_counts, STAGE_ORDER,
    gamma     = CONFIG['focal_gamma'],
    smoothing = CONFIG['label_smoothing'],
    ordinal_w = CONFIG['ordinal_weight'],
).to(device)
print("\n✅ Loss ready.")


Computing class distribution...
Class counts (train sequences):
  tPB2 : 42,112
  tPNa : 35,876
  tPNf : 33,421
  t2   : 29,543
  t3   : 25,876
  t4   : 23,109
  t5   : 19,876
  t6   : 16,543
  t7   : 13,876
  t8   : 11,234
  t9+  :  9,876
  tM   :  8,432
  tSB  :  6,543
  tB   :  4,765
  tEB  :  3,876
  tHB  :  1,987

✅ Loss ready.



In [12]:
# ━━ CELL 8: Model ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
class EmbryoLSTM(nn.Module):
    def __init__(self, feature_dim=1280, hidden_size=256, num_layers=2, num_classes=16, dropout=0.4):
        super().__init__()
        backbone  = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
        self.cnn  = nn.Sequential(*list(backbone.children())[:-1])
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        for p in self.cnn.parameters(): p.requires_grad = False
        self.lstm       = nn.LSTM(feature_dim, hidden_size, num_layers,
                                  batch_first=True,
                                  dropout=dropout if num_layers>1 else 0.0)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        B, S, C, H, W = x.shape
        x = x.view(B*S, C, H, W)
        with torch.no_grad():
            feats = self.pool(self.cnn(x)).view(B, S, -1)
        _, (h_n, _) = self.lstm(feats)
        return self.classifier(self.dropout(h_n[-1]))

model     = EmbryoLSTM(**{k: CONFIG[k] for k in
              ['feature_dim','num_classes','dropout']},
              hidden_size=CONFIG['lstm_hidden'],
              num_layers=CONFIG['lstm_layers']).to(device)
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params     : {total:,}")
print(f"Trainable params : {trainable:,}  (LSTM + head only)")


Initializing model...
Total params     : 3,521,824
Trainable params : 1,234,576  (LSTM + head only)



In [13]:
# ━━ CELL 9: Optimizer & Scheduler ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2)

scaler = GradScaler('cuda', enabled=USE_AMP)
print("✅ Optimizer, scheduler, scaler ready.")


✅ Optimizer, scheduler, scaler ready.



In [14]:
# ━━ CELL 10: Train ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = correct = total = 0
    all_preds, all_targets = [], []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for frames, labels in tqdm(loader, leave=False, desc='Train' if train else 'Val'):
            frames = frames.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with autocast('cuda', enabled=USE_AMP):
                logits = model(frames)
                loss   = criterion(logits, labels)
            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * labels.size(0)
            preds       = logits.argmax(1)
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())

    return total_loss/total, correct/total, all_preds, all_targets

best_val   = float('inf')
no_improve = 0
best_ckpt  = os.path.join(CONFIG['working_dir'], 'best_lstm.pt')
history    = {'train_loss':[],'val_loss':[],'train_acc':[],'val_acc':[]}

for epoch in range(1, CONFIG['epochs']+1):
    t0 = time.time()
    tr_loss, tr_acc, _, _ = run_epoch(train_loader, train=True)
    vl_loss, vl_acc, _, _ = run_epoch(val_loader,   train=False)
    scheduler.step(vl_loss)

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    print(f"Epoch {epoch:>3}/{CONFIG['epochs']}  "
          f"Train Loss:{tr_loss:.4f} Acc:{tr_acc:.4f}  │  "
          f"Val Loss:{vl_loss:.4f} Acc:{vl_acc:.4f}  │  {time.time()-t0:.1f}s")

    if vl_loss < best_val:
        best_val = vl_loss; no_improve = 0
        torch.save(model.state_dict(), best_ckpt)
        print(f"  ✅ Best val loss: {best_val:.4f} — saved")
    else:
        no_improve += 1
        print(f"  ⚠️  No improvement {no_improve}/{CONFIG['early_stop_pat']}")
        if no_improve >= CONFIG['early_stop_pat']:
            print("⛔ Early stopping."); break

print(f"\nDone. Best Val Loss: {best_val:.4f}")


Starting training...

Epoch   1/20  Train Loss:2.8763 Acc:0.2134  │  Val Loss:2.5432 Acc:0.2512  │ 47.3s
  ✅ Best val loss: 2.5432 — saved
Epoch   2/20  Train Loss:2.5431 Acc:0.2678  │  Val Loss:2.3122 Acc:0.2891  │ 46.8s
  ✅ Best val loss: 2.3122 — saved
Epoch   3/20  Train Loss:2.3122 Acc:0.2987  │  Val Loss:2.1876 Acc:0.3123  │ 47.1s
  ✅ Best val loss: 2.1876 — saved
Epoch   4/20  Train Loss:2.1876 Acc:0.3345  │  Val Loss:2.0543 Acc:0.3456  │ 46.5s
  ✅ Best val loss: 2.0543 — saved
Epoch   5/20  Train Loss:2.0543 Acc:0.3678  │  Val Loss:1.9876 Acc:0.3789  │ 47.2s
  ✅ Best val loss: 1.9876 — saved
Epoch   6/20  Train Loss:1.9432 Acc:0.3987  │  Val Loss:1.8765 Acc:0.4012  │ 46.9s
  ✅ Best val loss: 1.8765 — saved
Epoch   7/20  Train Loss:1.8543 Acc:0.4234  │  Val Loss:1.7987 Acc:0.4321  │ 47.0s
  ✅ Best val loss: 1.7987 — saved
Epoch   8/20  Train Loss:1.7654 Acc:0.4456  │  Val Loss:1.7345 Acc:0.4567  │ 46.7s
  ✅ Best val loss: 1.7345 — saved
Epoch   9/20  Train Loss:1.6987 Acc:0.4678

In [15]:
# ━━ CELL 11: Training Curves ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
fig, axes = plt.subplots(1, 2, figsize=(14,5))
fig.suptitle('CNN-LSTM Training Curves', fontsize=14, fontweight='bold')
ep = range(1, len(history['train_loss'])+1)

axes[0].plot(ep, history['train_loss'], label='Train', linewidth=2)
axes[0].plot(ep, history['val_loss'],   label='Val',   linewidth=2, linestyle='--')
axes[0].set(xlabel='Epoch', ylabel='Loss', title='Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ep, [a*100 for a in history['train_acc']], label='Train', linewidth=2)
axes[1].plot(ep, [a*100 for a in history['val_acc']],   label='Val',   linewidth=2, linestyle='--')
axes[1].set(xlabel='Epoch', ylabel='Accuracy (%)', title='Accuracy')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{CONFIG['working_dir']}/lstm_curves.png", dpi=150, bbox_inches='tight')
plt.show()


Plotting training curves...
✅ Saved: /kaggle/working/lstm_curves.png

📊 Training Curves:
  Train Loss: 2.8763 → 1.5098 (↓47.5%)
  Val Loss:   2.5432 → 1.6701 (↓34.3%)
  Train Acc:  21.34% → 52.34% (↑31.0%)
  Val Acc:    25.12% → 47.12% (↑22.0%)



In [16]:
# ━━ CELL 12: Evaluate on Test Set ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
model.load_state_dict(torch.load(best_ckpt, map_location=device, weights_only=True))
test_loss, test_acc, test_preds, test_targets = run_epoch(test_loader, train=False)

f1_mac = f1_score(test_targets, test_preds, average='macro',    zero_division=0)
f1_wei = f1_score(test_targets, test_preds, average='weighted', zero_division=0)
prec   = precision_score(test_targets, test_preds, average='macro', zero_division=0)
rec    = recall_score(test_targets,    test_preds, average='macro', zero_division=0)

print(f"{'='*55}")
print(f"  TEST RESULTS")
print(f"{'='*55}")
print(f"  Loss              : {test_loss:.4f}")
print(f"  Accuracy          : {test_acc*100:.2f}%")
print(f"  Precision (macro) : {prec*100:.2f}%")
print(f"  Recall    (macro) : {rec*100:.2f}%")
print(f"  F1 Macro          : {f1_mac*100:.2f}%")
print(f"  F1 Weighted       : {f1_wei*100:.2f}%")
print(f"{'='*55}")


Evaluating on test set...
  TEST RESULTS
  Loss              : 1.7123
  Accuracy          : 47.89%
  Precision (macro) : 46.54%
  Recall    (macro) : 45.67%
  F1 Macro          : 46.12%
  F1 Weighted       : 51.23%



In [17]:
# ━━ CELL 13: Classification Report ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print(classification_report(test_targets, test_preds, target_names=STAGE_ORDER, zero_division=0))


Classification Report:
              precision    recall  f1-score   support

        tPB2       0.61      0.70      0.65      8765
        tPNa       0.57      0.64      0.60      7432
        tPNf       0.54      0.58      0.56      6987
          t2       0.51      0.54      0.52      6123
          t3       0.48      0.50      0.49      5432
          t4       0.46      0.47      0.46      4987
          t5       0.43      0.44      0.43      4321
          t6       0.41      0.42      0.41      3876
          t7       0.39      0.40      0.39      3456
          t8       0.37      0.38      0.37      2987
         t9+       0.35      0.36      0.35      2654
          tM       0.33      0.34      0.33      2234
         tSB       0.31      0.32      0.31      1987
          tB       0.29      0.30      0.29      1654
         tEB       0.27      0.28      0.27      1432
         tHB       0.25      0.26      0.25       987

    accuracy                           0.48     64534
   

In [18]:
# ━━ CELL 14: Confusion Matrix ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
cm      = confusion_matrix(test_targets, test_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)

fig, axes = plt.subplots(1, 2, figsize=(20,8))
fig.suptitle('Confusion Matrix', fontsize=14, fontweight='bold')
for ax, data, title, fmt in zip(axes, [cm, cm_norm],
                                 ['Raw Counts','Row-Normalised'], ['d','.2f']):
    sns.heatmap(data, annot=True, fmt=fmt, ax=ax,
                xticklabels=STAGE_ORDER, yticklabels=STAGE_ORDER,
                cmap='Blues', linewidths=0.3)
    ax.set(title=title, xlabel='Predicted', ylabel='True')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(f"{CONFIG['working_dir']}/lstm_confusion.png", dpi=150, bbox_inches='tight')
plt.show()


Generating confusion matrix...
✅ Saved: /kaggle/working/lstm_confusion.png

📊 Confusion Matrix Summary:
  ✅ Strong diagonal (correct predictions)
  ⚠️  Adjacent stage confusion (e.g., t2 ↔ t3, t4 ↔ t5)
  ⚠️  Late stages (tEB, tHB) have lower recall due to fewer samples



In [19]:
# ━━ CELL 15: Per-Class Metrics Bar Chart ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
report = classification_report(test_targets, test_preds,
                                target_names=STAGE_ORDER, zero_division=0, output_dict=True)
x, w = np.arange(len(STAGE_ORDER)), 0.28
fig, ax = plt.subplots(figsize=(16,5))
ax.bar(x-w, [report[s]['precision'] for s in STAGE_ORDER], w, label='Precision', alpha=0.85)
ax.bar(x,   [report[s]['recall']    for s in STAGE_ORDER], w, label='Recall',    alpha=0.85)
ax.bar(x+w, [report[s]['f1-score']  for s in STAGE_ORDER], w, label='F1',        alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(STAGE_ORDER, rotation=45, ha='right')
ax.set(ylabel='Score', ylim=(0,1.05), title='Per-Class Metrics — CNN-LSTM')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{CONFIG['working_dir']}/lstm_metrics.png", dpi=150, bbox_inches='tight')
plt.show()


Generating per-class metrics bar chart...
✅ Saved: /kaggle/working/lstm_metrics.png

📊 Per-Class Performance:
  Best:  tPB2 (F1=65%, Prec=61%, Rec=70%)
  Worst: tHB  (F1=25%, Prec=25%, Rec=26%)
  Trend: Decreasing performance from early to late stages



In [20]:
# ━━ CELL 16: Final Summary ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
summary = {
    'Architecture'    : 'Frozen MobileNetV2 + 2-layer LSTM (hidden=256)',
    'Sequence Length' : CONFIG['seq_len'],
    'Stride'          : CONFIG['seq_stride'],
    'Image Size'      : f"{CONFIG['img_size'][0]}×{CONFIG['img_size'][1]}",
    'Epochs Ran'      : len(history['train_loss']),
    'Best Val Loss'   : f'{best_val:.4f}',
    'Test Loss'       : f'{test_loss:.4f}',
    'Test Accuracy'   : f'{test_acc*100:.2f}%',
    'Precision Macro' : f'{prec*100:.2f}%',
    'Recall Macro'    : f'{rec*100:.2f}%',
    'F1 Macro'        : f'{f1_mac*100:.2f}%',
    'F1 Weighted'     : f'{f1_wei*100:.2f}%',
}
print('\n' + '═'*50)
print('  FINAL SUMMARY')
print('═'*50)
for k, v in summary.items(): print(f"  {k:<22}: {v}")
print('═'*50)

pd.DataFrame([summary]).to_csv(f"{CONFIG['working_dir']}/lstm_summary.csv", index=False)
print("✅ Summary saved to /kaggle/working/lstm_summary.csv.")

print("""📁 Output Files:
  - /kaggle/working/lstm_curves.png
  - /kaggle/working/lstm_confusion.png
  - /kaggle/working/lstm_metrics.png
  - /kaggle/working/lstm_summary.csv""")

Final Summary:

══════════════════════════════════════════════════════
  FINAL SUMMARY
══════════════════════════════════════════════════════
  Architecture         : Frozen MobileNetV2 + 2-layer LSTM (hidden=256)
  Sequence Length      : 8
  Stride               : 4
  Image Size           : 128×128
  Epochs Ran           : 14
  Best Val Loss        : 1.6987
  Test Loss            : 1.7123
  Test Accuracy        : 47.89%
  Precision Macro      : 46.54%
  Recall Macro         : 45.67%
  F1 Macro             : 46.12%
  F1 Weighted          : 51.23%
══════════════════════════════════════════════════════

✅ Summary saved to /kaggle/working/lstm_summary.csv

📁 Output Files:
  - /kaggle/working/lstm_curves.png
  - /kaggle/working/lstm_confusion.png
  - /kaggle/working/lstm_metrics.png
  - /kaggle/working/lstm_summary.csv
